# Calculate canonical marker gene scores

In [ ]:
# eval "$(conda shell.bash hook)"
# conda init
# conda activate /work/islet_cartography_scrna/scrna_cartography_deseq
# python -m ipykernel install --user --name scrna_cartography_deseq --display-name "deseq"

#### Libraries

In [1]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pathlib import Path     # Object-oriented filesystem paths
from pyhere import here      # Reproducible project paths
import gc

# Single-cell data handling
import anndata as ad            # Core data structure for single-cell data
import scanpy as sc
import pyucell as uc
import gseapy as gp

# dataframes
import pandas as pd
import numpy as np

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import misc as mi
import diff_genes as dg

In [2]:
# Paths
base_dir = str(here('data/annotate/'))
plot_dir = os.path.join(base_dir, 'plot') 
files_dir = os.path.join(base_dir, 'files') 

anndata_dir = str(here('data/anndata/'))

#### Load

In [3]:
adata = ad.read_h5ad(os.path.join(anndata_dir, "AH_combined.h5ad"))

#### Generate list of marker genes

In [9]:
genes = {
    "acinar" : ['CTRB1', 'KLK1', 'RBPJL', 'PTF1A', 'CELA3A',
 'PRSS1',
 'SPINK1',
 'ZG16',
 'CEL',
 'CELA2A',
 'CPB1',
 'CELA1',
 'RNASE1',
 'AMY2B',
 'CPA2',
 'CPA1',
 'CELA3B',
 'PNLIP',
 'CTRB2',
 'PLA2G1B',
 'PRSS2',
 'CLPS',
 'REG1A',
 'SYCN',
 'PNLIPRP1',
 'CTRC',
 'REG3A',
 'SERPINA3',
 'PRSS3',
 'REG1B',
 'CFB',
 'GDF15',
 'MUC1',
 'C15ORF48',
 'DUOXA2',
 'AKR1C3',
 'OLFM4',
 'GSTA1',
 'LGALS2',
 'PDZK1IP1',
 'RARRES2',
 'CXCL17',
 'UBD',
 'GSTA2',
 'ANPEP',
 'LYZ',
 'ANGPTL4',
 'ALDOB'], 
    "acinar_i": ["RBPJL", "CHRM3", "LRIG1", "INSR", "FOXP2", "CHN2", "DTNA", "SDK1", "MAP3K5", "CAMK1D"], # https://doi.org/10.1053/j.gastro.2020.11.010
    "acinar_reg": ["REG3A", "REG3G", "REG1B", "REG1A"],
    "acinar_s": ["CPB1", "CPA1", "PRSS3", "PRSS1", "AMY2A", "CELA2A", "CELA3A", "CELA3B",
                 "CTRB1", "CTRB2", "CLPS", "PNLIP", "SPINK1", "CTRC", "CPA2", "ANXA4"],
    "endothelial" : ['PECAM1', 'ICAM1', 'ITGB3', 'SELE', 'VCAM1', 'MCAM', 'PROCR', 'TEK', 'FLT4', 'APLN', # Panglaodb canonical genes
               'VWF', 'NOS3', 'THBD', 'PLVAP', 'ACKR1', 'SLCO1C1', 'TMEM100', 'ADGRF5', 'ABCG2',
               'PODXL', 'NOSTRIN', 'MFSD2A', 'ACVRL1', 'AQP1', 'MYLK', 'RASIP1', 'FLI1', 'TIE1',
               'APLNR', 'NRP2', 'ADAMTS1', 'RPRM', 'FABP4', 'GPIHBP1', 'FHL2', 'LOX', 'KLK1',
               'ARHGEF15', 'CARD10', 'CLEC14A', 'DLL4', 'ESM1', 'GIMAP5', 'MMRN2', 'NOTCH4', 'NPR1',
               'PRKCH', 'RASGRP3', 'ROBO4', 'SCARF1', 'SOX18', 'SOX7', 'SPNS2', 'THSD1', 'APOLD1',
               'EMP1', 'CD36', 'RNASE1', 'CTGF', 'HYAL2', 'CLEC4G', 'GPR182', 'F8', 'RBP7', 'CALCRL',
               'FOXF1', 'CASZ1', 'AQP7', 'TCF15', 'CD300LG', 'BTNL9', 'MEOX2', 'ERG', 'HEXIM1',
               'GLYCAM1', 'CD55', 'MMRN1', 'C7', 'RAMP3', 'VEGFC', 'GJA5', 'HEY1', 'RND1', 'BDP1',
               'CD46', 'MEOX1', 'CCL19', 'MADCAM1', 'CYP1B1', 'IRX3', 'BIRC2', 'LYVE1', 'SEMA3D',
               'EMCN', 'WFDC1', 'ADGRL4', 'VWA1', 'ECE1', 'PTPRB', 'CLDN5', 'TBX1', 'SEMA7A',
               'FOXF2', 'PDGFB', 'ECSCR', 'ELK3', 'CDH5', 'PLEC', 'STAB1', 'TGFBR2', 'CD93', 'CXCL1',
               'RGS5', 'SLC7A5', 'ENG', 'KDR', 'SLC2A1', 'EGFL7', 'FLT1', 'EPAS1', 'EDNRB', 'KCNJ8',
               'CD82', 'CHST1', 'PLAC8', 'TSPAN8', 'ETS1', 'CD34', 'PDPN', 'PROX1', 'EHD3', 'SRGN',
               'S100A10', 'CLIC4', 'USHBP1', 'MYF6', 'OIT3', 'IL1A', 'BMP2', 'C1QTNF1', 'PCDH12',
               'DPP4', 'IGFBP7', 'PALMD', 'POSTN', 'BMX', 'SLC38A5', 'XDH', 'SPARC', 'MGLL',
               'SLC9A3R2', 'RGCC', 'ICAM2', 'MGP', 'SPARCL1', 'TM4SF1', 'ID1', 'ADIRF', 'CD9',
               'SRPX', 'ID3', 'CAV1', 'GNG11', 'HSPG2', 'CCL14', 'CLEC1B', 'FCN2', 'S100A13',
               'FCN3', 'CRHBP', 'IFI27', 'CCL23', 'SGK1', 'DNASE1L3', 'LIFR', 'PCAT19', 'CDKN1C',
               'INMT', 'PTGDS', 'TIMP3', 'GPM6A', 'FAM167B', 'LTC4S', 'STAB2'],  
    "islet_endothelial": ["ACE", "PASK", "F2RL3", "ESM1", "CXCR4", "ACKR3", "UNC5B",
                           "LAMA4", "CREM", "COL13A1", "NKX2-3", "ANGPTL2", "THBS1"], # https://www.nature.com/articles/s41467-024-55415-3
    "acinar_endothelial": ["AQP1", "CCL14", "JUN", "FOS", "FOSB", "CD74", "KLF4", "KLF2",
                            "ATOH8", "GPIHBP1"],
    "pericyte" : ["LAMA2", "NG2", # https://www.nature.com/articles/s41598-021-81774-8 
                  'PECAM1', 'PDGFRB', 'CSPG4', 'ANPEP', 'ACTA2', 'DES', 'RGS5', 'ABCC9', 'KCNJ8', 'CD248', # Panglaodb canonical genes
                  'DLK1', 'TEK', 'NOTCH3', 'ANGPT1', 'ZIC1', 'HIGD1B', 'MCAM', 'COLEC11', 'VTN', 'STEAP4',
                  'ATP13A5', 'AOC3', 'ANGPT2', 'INPP4B', 'VIM', 'PTH1R', 'IFITM1', 'TBX18', 'NT5E', 'MFGE8',
                  'ALPL', 'COL1A1', 'MYO1B', 'COG7', 'P2RY14', 'HEYL', 'GNB4', 'MSX1', 'CTGF'], 
    "ductal_mucin": ["CDKN1A", "ADIRF", "ANXA1", "FOXC1", "PLAT", "TMPRSS4", "SEN", "GPRC5A", # Baron paper
                      "MUC20", "CXCL17", "AREG", "ADGRF1", "CEACAM", "CLDN", "IGFBP3", "DHRS9",
                      "LCN2", "SLPI", "LF3", "TXNIP", "MGST1", "CLDN4", "SDC1", "MUC1", "AKR1C3",
                      "TM4SF4", "PMP3", "SPINK1", "TSPAN8", "ASS1", "APOE", "ALDH2", "ALOX5",
                      "GDA", "DUOX2", "DUOXA2", "LGALS3", "TMEM176A", "TFF2", "CALM4", "TM9SF4",
                      "PDGFR", "CDH19", "FGF19", "ID2", "LYZ", "GSTA1", "CXCL3", "UCA1", "AKR1B10"],
    "ductal" : ['CFTR', 'HNF1B', 'KRT20', 'MUC1', 'AMBP', 'HHEX', 'ANXA4', 'SPP1', 'PDX1', 'SERPINA3', # Panglaodb canonical genes
                'CFB', 'GDF15', 'AKR1C3', 'MMP7', 'DEFB1', 'SERPING1', 'TSPAN8', 'CLDN10', 'SLPI',
                'SERPINA5', 'PIGR', 'CLDN1', 'LGALS4', 'PERP', 'PDLIM3', 'WFDC2', 'SLC3A1', 'AQP1',
                'ALDH1A3', 'VTCN1', 'KRT19', 'TFF1', 'TFF2', 'KRT7', 'CLDN4', 'LAMB3', 'TACSTD2',
                'CCL2', 'DCDC2', 'CXCL2', 'CTSH', 'S100A10'],
    "schwann" : ["NGFR", "CDH19", "UCN2", "SOX10", "S100A1", "PLP1", "TSPAN11", "WNT16", "SOX2", "TFAP2A"], # Azimuth
    "stellate" : ['RGS5', 'PDGFRA', 'INHBA', 'TGFB1', 'COL6A1', 'MMP11', 'FN1', 'COL4A1', 'COL6A2',
            'MGP', 'COL3A1', 'SPARC', 'COL1A1', 'TIMP3', 'TNFAIP6', 'COL1A2', 'SFRP2',
            'COL6A3', 'THY1'],
    "stellate_q" : ['RGS5', 'C11orf96', 'FABP4', 'CSRP2', 'IL24', 'ADIRF', 'NDUFA4L2', 'GPX3', 'IGFBP4', 'ESAM', 'CYGB', 'PLIN2'], # Azimuth and papers
    "stellate_a" : ['COL1A1', 'COL1A2', 'COL6A3', 'COL3A1', 'TIMP3', 'TIMP1', 'CTHRC1', 'SFRP2', 'BGN', 'LUM', 
                    'ACAT2', "MMP3", "VCAN", "ABEP1", "TIMP1"], # Azimuth and papers
    "myeloid" : ['MARCO', 'APOE', 'CD14', 'HLA-DRA', 'SPP1', 'LY6E', 'C1QC', 'CCR2', 'FCN1',  # From review DOI: 10.3389/fonc.2022.881871
                  'CLEC9A', 'BATF3', 'IRF8', 'IDO1', 'XCR1', 'CD1C', 'FCER1A', 'TCR7', 'IRF7',
                  'GZMB', 'LILRA4', 'CD207', 'CD1A1', 'GOS2', 'FCGR3B', 'S100A8', 'CXCR2'],
    "mast" : ["CD16", "CD32", "CD34", "CD63", "ENPP3", "FCER1", "ITGA4", "ITGB7", "KIT", "VCAM1", # From biocompare https://www.biocompare.com/Editorial-Articles/581481-A-Guide-to-Mast-Cell-Markers/
             "KIT", "TPSAB1", "CPA3", "HDC"], # generally expressed as common markers in literature 
    "cycling": ["UBE2C", "TOP2A", "CDK1", "BIRC5", "PBK", "CDKN3", "MKI67", "CDC20", "CCNB2", "CDCA3"], #Azimuth
    # All of these are from van gurp
     "alpha": ['TMEM176A', 'SLC7A2', 'FAP', 'FXYD5', 'FXYD3', 'SMIM24', 'PLIN3', 'TMEM176B', 'GCG',
              'GLS', 'PAPPA2', 'RGS4', 'TTR', 'CYSTM1', 'CLU', 'PCSK2', 'F10', 'VGF', 'PALLD',
              'PEMT', 'CFC1', 'LOXL4', 'PLCE1', 'GC', 'ITGB1', 'CRYBA2', 'ALDH1A1', 'B2M', 'SPINT2',
              'TM4SF4', 'IRX2', 'NAA20', 'MUC13', 'KCTD12', 'HIGD1A', 'DPP4', 'HLA-E', 'HLA-A',
              'GPX3', 'RNASEK'],
    "beta": ['ABCC8', 'SYT13', 'RRAGD', 'TIMP2', 'CASR', 'HERPUD1', 'TGFBR3', 'PFN2', 'ATP2A3',
             'ENO1', 'WSCD2', 'TNS1', 'SCGN', 'ARG2', 'ERO1B', 'GNAS', 'PEBP1', 'KCNK16',
             'HSP90AB1', 'ITPR3', 'SCD', 'RPL3', 'DHRS7', 'DHRS2', 'SCG3', 'SLC39A14', 'STX1A',
             'NPTX2', 'TSPAN13', 'PRUNE2', 'GLIS3', 'HSPA8', 'LDHB', 'SERINC1', 'PERP', 'PHACTR2',
             'VEGFA', 'RPL24', 'PRDX1', 'TSPAN1', 'IAPP', 'RPL5', 'PFKFB2', 'CDKN1A', 'RPL23',
             'DNAJB9', 'CDKN1C', 'INS-IGF2', 'C1QL1', 'SLC6A6', 'MAP1B', 'MTUS2', 'PEMT', 'YWHAQ',
             'PPP1R1A', 'GAD2', 'RPS6', 'SORL1', 'FXYD2', 'RBP4', 'HADH', 'PDX1', 'WARS1',
             'ADCYAP1', 'RGS16', 'SUSD4', 'TPM3', 'ARL6IP5', 'SCD5', 'OTULINL', 'SHISAL2B', 'G3BP1',
             'RPL7', 'GSN', 'SURF4', 'RPL7A', 'RPS3', 'ALDOA', 'G6PC2', 'UCHL1', 'CYP2U1', 'DBI',
             'ELMO1', 'EIF4A2', 'NKX6-1', 'SLC30A8', 'CYYR1', 'IGF2', 'ENTPD3', 'P2RY1', 'ALCAM',
             'TMEM37', 'PTEN', 'CNP', 'RPL4', 'CIROZ', 'PCSK1', 'NECTIN3', 'SELENOW', 'MXRA7',
             'MAFA', 'PLCXD3', 'CADM1', 'ROBO2', 'DLK1', 'RPS23', 'SAMD11', 'PSAP', 'RPS4X',
             'PAPSS2', 'MT-CYB', 'MT-ND2', 'MT-CO1', 'MT-ND3', 'MT-ND4', 'MT-ND1', 'MT-CO3',
             'MAFB', 'TMEM150C', 'INS', 'RPL17'],

    "delta": ['ETV1', 'BAIAP3', 'CD9', 'ISL1', 'PLEKHB1', 'VIM', 'ANK1', 'TIMP2', 'CASR', 'SYNE2',
              'CBLN4', 'VMP1', 'ABCC9', 'NLRP1', 'SLC17A6', 'DHRS2', 'NDRG4', 'CALB1', 'NPTX2',
              'UNC5B', 'MDK', 'SLC38A1', 'NCOA7', 'ARFGEF3', 'DPYSL3', 'BCHE', 'MLPH', 'PAPPA2',
              'OPRD1', 'LEPR', 'PRG4', 'RGS2', 'ADGRL2', 'BHLHE41', 'EDN3', 'MTUS1', 'AKAP12',
              'PDLIM4', 'LDHA', 'EHF', 'PKIB', 'SORL1', 'RBP4', 'HADH', 'AMIGO2', 'SCD5', 'HHEX',
              'CPB1', 'ABI3BP', 'CXADR', 'SST', 'TPPP3', 'DIRAS3', 'FRZB', 'PSIP1', 'AQP3', 'LRFN5',
              'GABRB3', 'SEC11C', 'MS4A8', 'EEIG1', 'RASSF6', 'PCSK1', 'ERBB4', 'CADM1', 'PCP4',
              'FFAR4', 'EYS', 'SERPINA1', 'S100A6', 'F5', 'TMSB4X', 'C22orf42', 'GPX3', 'TENM3'],

    "gamma": ['ARX', 'THSD7A', 'ETV1', 'PAX6', 'ABCC9', 'FGFR1', 'SLC4A4', 'ABCB1', 'CALB1',
              'STMN2', 'CHN2', 'TMEM176B', 'SLC6A4', 'PPY', 'DPYSL3', 'ID2', 'SCGB2A1', 'ID1',
              'AKAP12', 'CHRM3', 'MEIS2', 'FXYD2', 'GCNT3', 'GC', 'TMEM47', 'SPOCK1', 'CPB1',
              'SPINK1', 'CARTPT', 'AQP3', 'AMOTL1', 'BMERB1', 'PXK', 'TM4SF4', 'SEMA3E', 'SCG2',
              'ID4', 'SERTM1', 'PTP4A3', 'SLITRK6', 'S100A10', 'INPP5F', 'PEG10', 'FXYD6-FXYD2',
              'TXNIP'],
    "epsilon" : ["BHMT", "VSTM2L", "PHGR1", "TM4SF5", "ANXA13", "ASGR1", "DEFB1", "GHRL", "COL22A1", "OLFML3", "ACSL1", "FRZB", "PHGR1"] # azimuth and other studies
    
}

#### Compute UCell

In [10]:
uc.compute_ucell_scores(adata, signatures=genes, suffix='', missing_genes = "skip")
adata.obs[['cell_type', 'cell_type_broad', 'ic_id_dataset'] + list(genes.keys())].to_csv(os.path.join(files_dir, "ucell_canonical_genes.csv"))

Access: 1st of september 2026

In [ ]:
! aria2c -d /work/islet_cartography_scrna/data/annotate/files/ -o panglaodb.tsv.gz https://panglaodb.se/markers/PanglaoDB_markers_27_Mar_2020.tsv.gz

In [4]:
panglaodb_genes = pd.read_csv(os.path.join(files_dir, "panglaodb.tsv.gz"), sep = "\t")

In [5]:
panglaodb_genes_flt = panglaodb_genes[panglaodb_genes['species'].isin(['Mm Hs', 'Hs'])]
panglaodb_genes_flt = panglaodb_genes_flt[panglaodb_genes_flt['organ'].isin(['Pancreas', 'Immune system', 'Vasculature'])]
panglaodb_genes_flt = panglaodb_genes_flt[panglaodb_genes_flt['canonical marker'].isin([1.0])]

In [6]:
panglaodb_dict = (
    panglaodb_genes_flt
    .groupby('cell type')['official gene symbol']
    .apply(list)
    .to_dict()
)

#### Compute ucell score

In [ ]:
uc.compute_ucell_scores(adata, signatures=panglaodb_dict, suffix='', missing_genes = "skip")
adata.obs[['cell_type', 'cell_type_broad'] + list(panglaodb_dict.keys())].to_csv(os.path.join(files_dir, "panglaodb_ucell.csv"))

#### Van grup genes
Access 1st of steptember 2026

In [ ]:
! aria2c -d /work/islet_cartography_scrna/data/annotate/files/ -o gurp_alpha.tsv "https://www.gsea-msigdb.org/gsea/msigdb/human/download_geneset.jsp?geneSetName=VANGURP_PANCREATIC_ALPHA_CELL&fileType=TSV"
! aria2c -d /work/islet_cartography_scrna/data/annotate/files/ -o gurp_beta.tsv "https://www.gsea-msigdb.org/gsea/msigdb/human/download_geneset.jsp?geneSetName=VANGURP_PANCREATIC_BETA_CELL&fileType=TSV"
! aria2c -d /work/islet_cartography_scrna/data/annotate/files/ -o gurp_delta.tsv "https://www.gsea-msigdb.org/gsea/msigdb/human/download_geneset.jsp?geneSetName=VANGURP_PANCREATIC_DELTA_CELL&fileType=TSV"
! aria2c -d /work/islet_cartography_scrna/data/annotate/files/ -o gurp_gamma.tsv "https://www.gsea-msigdb.org/gsea/msigdb/human/download_geneset.jsp?geneSetName=VANGURP_PANCREATIC_GAMMA_CELL&fileType=TSV"

In [ ]:
files = {
    "alpha": "/work/islet_cartography_scrna/data/annotate/files/gurp_alpha.tsv",
    "beta":  "/work/islet_cartography_scrna/data/annotate/files/gurp_beta.tsv",
    "delta": "/work/islet_cartography_scrna/data/annotate/files/gurp_delta.tsv",
    "gamma": "/work/islet_cartography_scrna/data/annotate/files/gurp_gamma.tsv",
}

vangurp_dict = {}

for cell_type, path in files.items():
    df = pd.read_csv(path, sep="\t", header=None)

    # Case 1: MSigDB metadata-style export (key-value rows)
    # Look for a row where the first column is GENE_SYMBOLS
    gene_row = df[df[0] == "GENE_SYMBOLS"]
    if not gene_row.empty:
        genes = gene_row.iloc[0, 1].split(",")
    else:
        # Case 2: simple one-gene-per-line file
        genes = df[0].dropna().tolist()

    vangurp_dict[f"{cell_type}"] = [g.strip() for g in genes if g.strip()]

print({k: len(v) for k, v in vangurp_dict.items()})

In [ ]:
vangurp_dict

In [ ]:
uc.compute_ucell_scores(adata, signatures=vangurp_dict, suffix='', missing_genes = "skip")
adata.obs[['cell_type', 'cell_type_broad'] + list(vangurp_dict.keys())].to_csv(os.path.join(files_dir, "vangurp_dict_ucell.csv"))